In [1]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('soil-grain-size-from-photos')

print("Path to competition files:", path)

c:\Users\axeld\VSCode\Kaggle\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to competition files: C:\Users\axeld\.cache\kagglehub\competitions\soil-grain-size-from-photos


In [2]:
# import random
# import matplotlib.pyplot as plt
# from pathlib import Path
# from PIL import Image
# 
# image_dir = Path(f"{path}/Training-All_Photos_updated/Training-All_Photos_updated")
# 
# image_paths = [
#     p for p in image_dir.rglob("*") if p.suffix.lower() in set([".jpg", ".jpeg", ".png", ".bpm", ".webp"])
# ]
# 
# print(f"{len(image_paths)} images in {image_dir.resolve()}")
# 
# sample = random.sample(image_paths, min(9, len(image_paths)))
# 
# if sample:
#     fig, axes = plt.subplots(3, 3, figsize=(12, 12))
# 
#     for ax in axes.flat:
#         ax.axis("off")
# 
#     for ax, image_path in zip(axes.flat, sample):
#         with Image.open(image_path) as img:
#             ax.imshow(img.convert("RGB"))
#         ax.set_title(image_path.name, fontsize=9)
# 
#     plt.tight_layout()
#     plt.show()

In [3]:
import pandas as pd
from pathlib import Path

data_dir = Path(path)

train_labels = pd.read_csv(data_dir / "Training_labels_updated.csv")
train_image_dir = data_dir / "Training-All_Photos_updated/Training-All_Photos_updated"

image_paths = sorted(
    p for p in train_image_dir.rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
)

images = pd.DataFrame({
    "image_path": image_paths,
    "sample_id": [p.stem.rsplit("_", 2)[-2] for p in image_paths]
})

image_table = images.merge(
    train_labels,
    on="sample_id",
    how="left",
    validate="many_to_one",
    indicator=True
)
image_table = image_table.drop(columns="_merge")

target_cols = [c for c in train_labels.columns if c != "sample_id"]

print(f"Target cols: {target_cols}")

image_table.head()


Target cols: ['0.002', '0.0063', '0.02', '0.063', '0.2', '0.63', '2', '6.3', '20', '63', '200']


,image_path,sample_id,0.002,0.0063,0.02,0.063,0.2,0.63,2,6.3,20,63,200
0,C:\Users\axeld\.cache\kagglehub\competitions\s...,H374,1.1223,2.4374,5.5348,9.3031,13.1208,18.2481,29.4152,51.2113,76.4875,92.6403,100.0
1,C:\Users\axeld\.cache\kagglehub\competitions\s...,H374,1.1223,2.4374,5.5348,9.3031,13.1208,18.2481,29.4152,51.2113,76.4875,92.6403,100.0
2,C:\Users\axeld\.cache\kagglehub\competitions\s...,H374,1.1223,2.4374,5.5348,9.3031,13.1208,18.2481,29.4152,51.2113,76.4875,92.6403,100.0
3,C:\Users\axeld\.cache\kagglehub\competitions\s...,F827,9.4904,19.3892,47.6257,89.5554,99.8965,99.9896,100.0000,100.0000,100.0000,100.0000,100.0
4,C:\Users\axeld\.cache\kagglehub\competitions\s...,F827,9.4904,19.3892,47.6257,89.5554,99.8965,99.9896,100.0000,100.0000,100.0000,100.0000,100.0


In [4]:
import torch
from PIL import Image
from torch.utils.data import Dataset

class SoilDataset(Dataset):
    def __init__(self, table, target_cols, transform):
        self.table = table.reset_index(drop=True)
        self.targets = torch.tensor(
            self.table[target_cols].to_numpy(dtype="float32") / 100.0
        )
        self.transform = transform

    def __len__(self):
        return len(self.table)

    def __getitem__(self, idx):
        with Image.open(self.table.loc[idx, "image_path"]) as image:
            image = self.transform(image.convert("RGB"))

        return image, self.targets[idx]

In [5]:
import torch

torch.manual_seed(42)

device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
device

'cuda'

In [ ]:
import torch.nn as nn

class SoilModelBaseline(nn.Module):
    def __init__(self, n_categories: int, dropout: float):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(32, 64, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(64, 128, kernel_size=3),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1, 1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(128, n_categories)
        )

    def forward(self, X):
        x = self.features(X)
        return self.classifier(x)

In [6]:
@torch.no_grad()
def evaluate(model, val_loader, criterion, metric):
    model.eval()
    metric.reset()

    total_loss = 0.0
    total_samples = 0
    for X, y in val_loader:
        X = X.to(device)
        y = y.to(device)

        predictions = model(X)
        loss = criterion(predictions, y)
        metric.update(predictions, y)

        total_loss += loss.item() * len(y)
        total_samples += len(y)

    return {
        "loss": total_loss / total_samples,
        "metric": metric.compute().item()
    }

def train_one_epoch(model, train_loader, val_loader, criterion, optimizer, metric):
    model.train()
    metric.reset()

    total_loss = 0.0
    total_samples = 0
    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        predictions = model(X)
        loss = criterion(predictions, y)

        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(y)
        total_samples += len(y)

    train_loss = total_loss / total_samples
    val = evaluate(model, val_loader, criterion, metric)

    return (train_loss, val["loss"], val["metric"])

def fit(model, train_loader, val_loader, criterion, optimizer, metric, n_epochs):
    history = {
        "train_losses": [],
        "val_losses": [],
        "val_metrics": []
    }

    for epoch in range(n_epochs):
        train_loss, val_loss, val_metric = train_one_epoch(
            model, train_loader, val_loader,
            criterion, optimizer, metric
        )
        history["train_losses"].append(train_loss)
        history["val_losses"].append(val_loss)
        history["val_metrics"].append(val_metric)

        print(
            f"Epoch {epoch + 1}: "
            f"Train loss: {train_loss} | ",
            f"Val loss: {val_loss} | ",
            f"Val metric: {val_metric}"
        )

In [7]:
from sklearn.model_selection import GroupKFold
from torch.utils.data import DataLoader
from torchvision import transforms

cv = GroupKFold(n_splits=5)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

for fold, (train_idx, val_idx) in enumerate(
    cv.split(image_table, groups=image_table["sample_id"]),
    start=1
):
    torch.manual_seed(42 + fold)
    train_table = image_table.iloc[train_idx].reset_index(drop=True)
    val_table = image_table.iloc[val_idx].reset_index(drop=True)

    train_loader = DataLoader(
        SoilDataset(train_table, target_cols, transform),
        batch_size=32, 
        shuffle=True
    )
    val_loader = DataLoader(
        SoilDataset(val_table, target_cols, transform),
        batch_size=32,
        shuffle=False
    )

    print(
        f"Fold {fold}: "
        f"{train_table["sample_id"].nunique()} training samples, "
        f"{val_table["sample_id"].nunique()} val samples"
    )

Fold 1: 19 training samples, 5 val samples
Fold 2: 19 training samples, 5 val samples
Fold 3: 20 training samples, 4 val samples
Fold 4: 19 training samples, 5 val samples
Fold 5: 19 training samples, 5 val samples
